# IMPORT BIBLIOTEK

In [41]:
import mysql.connector
import pandas as pd
import random
import numpy as np
from datetime import datetime, timedelta


# POŁĄCZENIE Z BAZĄ DANYCH

In [43]:

con = mysql.connector.connect(
    host = "giniewicz.it",
    user = "team07",
    password = "te@mlot",
    database = "team07"
)

if(con):
    print("Połączenie udane")
else:
    print("Połączenie nieudane")

Połączenie udane


# TWORZENIE SCHEMATU BAZY DANYCH

In [44]:
cursor = con.cursor()

schema_sql = """
-- Wyłączenie sprawdzania kluczy obcych
SET FOREIGN_KEY_CHECKS = 0;

-- Drop tables if they already exist
DROP TABLE IF EXISTS TripCrew;
DROP TABLE IF EXISTS TripParticipants;
DROP TABLE IF EXISTS Transactions;
DROP TABLE IF EXISTS RecoveryTasks;
DROP TABLE IF EXISTS RepairTasks;
DROP TABLE IF EXISTS Trips;
DROP TABLE IF EXISTS Destinations;
DROP TABLE IF EXISTS Employees;
DROP TABLE IF EXISTS Spaceships;
DROP TABLE IF EXISTS Rockets;
DROP TABLE IF EXISTS Customers;

-- Włączenie sprawdzania kluczy obcych
SET FOREIGN_KEY_CHECKS = 1;

-- Create Customers table
CREATE TABLE Customers (
    pesel VARCHAR(11) PRIMARY KEY,
    first_name VARCHAR(100),
    last_name VARCHAR(100),
    date_of_birth DATE,
    sex CHAR(1),
    is_insured BOOLEAN,
    email VARCHAR(150),
    phone_number VARCHAR(30),
    emergency_contact_name VARCHAR(100),
    emergency_contact_phone VARCHAR(30),
    emergency_contact_relation VARCHAR(50)
);

-- Create Rockets table
CREATE TABLE Rockets (
    rocket_id INTEGER PRIMARY KEY,
    name VARCHAR(100),
    type VARCHAR(100),
    manufacturing_date DATE,
    status VARCHAR(50),
    initial_cost DECIMAL(12,2)
);

-- Create Spaceships table
CREATE TABLE Spaceships (
    spaceship_id INTEGER PRIMARY KEY,
    name VARCHAR(100),
    type VARCHAR(100),
    capacity INTEGER,
    manufacturing_date DATE,
    status VARCHAR(50),
    initial_cost DECIMAL(12,2),
    travel_speed FLOAT
);

-- Create Employees table
CREATE TABLE Employees (
    pesel VARCHAR(11) PRIMARY KEY,
    first_name VARCHAR(100),
    last_name VARCHAR(100),
    sex CHAR(1),
    date_of_birth DATE,
    role VARCHAR(100),
    specialization VARCHAR(100),
    salary DECIMAL(10,2)
);

-- Create Destinations table
CREATE TABLE Destinations (
    destination_id INTEGER PRIMARY KEY,
    name VARCHAR(100),
    description TEXT,
    distance_range VARCHAR(100)
);

-- Create Trips table
CREATE TABLE Trips (
    trip_id INTEGER AUTO_INCREMENT PRIMARY KEY,
    name VARCHAR(100),
    description TEXT,
    destination_id INTEGER,
    launch_date DATE,
    duration_of_stay INTEGER,
    return_date DATE,
    spaceship_id INTEGER,
    rocket_id INTEGER,
    status VARCHAR(50),
    organization_cost DECIMAL(12,2),
    FOREIGN KEY (destination_id) REFERENCES Destinations(destination_id),
    FOREIGN KEY (spaceship_id) REFERENCES Spaceships(spaceship_id),
    FOREIGN KEY (rocket_id) REFERENCES Rockets(rocket_id)
);

-- Create TripParticipants table
CREATE TABLE TripParticipants (
    participant_id INTEGER AUTO_INCREMENT PRIMARY KEY,
    trip_id INTEGER,
    pesel VARCHAR(11),
    ticket_price DECIMAL(10,2),
    status VARCHAR(20),
    satisfaction_level FLOAT,
    FOREIGN KEY (trip_id) REFERENCES Trips(trip_id),
    FOREIGN KEY (pesel) REFERENCES Customers(pesel)
);

-- Create Transactions table
CREATE TABLE Transactions (
    transaction_id INTEGER PRIMARY KEY,
    customer_id VARCHAR(11),
    amount DECIMAL(10,2),
    transaction_date TIMESTAMP,
    method VARCHAR(50),
    FOREIGN KEY (customer_id) REFERENCES Customers(pesel)
);

-- Create TripCrew table
CREATE TABLE TripCrew (
    trip_crew_id INTEGER AUTO_INCREMENT PRIMARY KEY,
    trip_id INTEGER,
    employee_id VARCHAR(11),
    role VARCHAR(50),
    FOREIGN KEY (trip_id) REFERENCES Trips(trip_id),
    FOREIGN KEY (employee_id) REFERENCES Employees(pesel)
);

-- Create RecoveryTasks table
CREATE TABLE RecoveryTasks (
    recovery_task_id INTEGER AUTO_INCREMENT PRIMARY KEY,
    pesel VARCHAR(11),
    rocket_id INTEGER,
    start_date DATE,
    end_date DATE,
    status VARCHAR(50),
    FOREIGN KEY (pesel) REFERENCES Employees(pesel),
    FOREIGN KEY (rocket_id) REFERENCES Rockets(rocket_id)
);

-- Create RepairTasks table
CREATE TABLE RepairTasks (
    repair_task_id INTEGER AUTO_INCREMENT PRIMARY KEY,
    employee_id VARCHAR(11),
    spaceship_id INTEGER,
    start_date DATE,
    end_date DATE,
    status VARCHAR(50),
    FOREIGN KEY (employee_id) REFERENCES Employees(pesel),
    FOREIGN KEY (spaceship_id) REFERENCES Spaceships(spaceship_id)
);
"""
for statement in schema_sql.strip().split(';'):
    if statement.strip():
        cursor.execute(statement + ';')

con.commit()
print("Schemat bazy danych został utworzony.")

Schemat bazy danych został utworzony.


### DODANIE CELÓW PODRÓŻY

In [45]:
cursor = con.cursor()

schema_sql = """
-- Wstawianie danych do tabeli Destinations
INSERT INTO Destinations (destination_id, name, description, distance_range)
VALUES
  (1, 'Mercury', 'Najbliższa Słońcu planeta, znana z ekstremalnych temperatur i surowego krajobrazu.', '0.61-1.47 AU'),
  (2, 'Venus', 'Gorąca planeta z gęstą atmosferą, często nazywana bliźniaczką Ziemi.', '0.27-1.73 AU'),
  (3, 'Mars', 'Czerwona planeta z potencjalnymi śladami wody i celem przyszłych misji kolonizacyjnych.', '0.52-2.67 AU'),
  (4, 'Moon', 'Naturalny satelita Ziemi, oferujący spektakularne widoki i historyczne miejsca lądowań.', '0.00257 AU'),
  (5, 'Space Station', 'Orbitująca stacja kosmiczna na niskiej orbicie ziemskiej, idealna dla budżetowych wycieczek.', '0.00000267 AU');
"""
for statement in schema_sql.strip().split(';'):
    if statement.strip():
        cursor.execute(statement + ';')

con.commit()
print("Dane do tabeli Destinations zostały wstawione.")

Dane do tabeli Destinations zostały wstawione.


### Dodanie rakiet i statków kosmicznych

In [ ]:
cursor = con.cursor()

schema_sql = """
-- Usuwanie istniejących danych z tabel
DELETE FROM Rockets;
DELETE FROM Spaceships;

-- Wstawianie danych do tabeli Rockets
INSERT INTO Rockets (rocket_id, name,type, manufacturing_date, status, initial_cost)
VALUES
  (1, 'OrbitalLift-1','orbital', '2165-03-15', 'Active', 5000000.00),
  (2, 'OrbitalLift-2','orbital', '2165-04-20', 'Active', 5000000.00),
  (3, 'OrbitalLift-3','orbital', '2165-05-25', 'Active', 5000000.00),
  (4, 'OrbitalLift-4','orbital', '2165-06-30', 'Active', 5000000.00),
  (5, 'LunarExplorer-1','lunar', '2166-01-10', 'Active', 15000000.00),
  (6, 'LunarExplorer-2','lunar', '2166-02-15', 'Active', 15000000.00),
  (7, 'LunarExplorer-3','lunar', '2166-03-20', 'Active', 15000000.00),
  (8, 'LunarExplorer-4','lunar', '2166-04-25', 'Active', 15000000.00),
  (9, 'InterplanetaryVoyager-1','interplanetar', '2167-01-15', 'Active', 50000000.00),
  (10, 'InterplanetaryVoyager-2','interplanetar', '2167-02-20', 'Active', 50000000.00),
  (11, 'InterplanetaryVoyager-3','interplanetar', '2167-03-25', 'Active', 50000000.00),
  (12, 'InterplanetaryVoyager-4','interplanetar', '2167-04-30', 'Active', 50000000.00);

-- Wstawianie danych do tabeli Spaceships
INSERT INTO Spaceships (spaceship_id, name,type, capacity, travel_speed, manufacturing_date, status, initial_cost)
VALUES
  (1, 'OrbitalShuttle-1','orbital', 5, 7.8, '2165-03-20', 'Active', 3000000.00),
  (2, 'OrbitalShuttle-2','orbital', 5, 7.8, '2165-04-25', 'Active', 3000000.00),
  (3, 'OrbitalShuttle-3','orbital', 5, 7.8, '2165-05-30', 'Active', 3000000.00),
  (4, 'OrbitalShuttle-4','orbital', 5, 7.8, '2165-06-15', 'Active', 3000000.00),
  (5, 'LunarCruiser-1','lunar', 10, 10.5, '2166-01-20', 'Active', 12000000.00),
  (6, 'LunarCruiser-2','lunar', 10, 10.5, '2166-02-25', 'Active', 12000000.00),
  (7, 'LunarCruiser-3','lunar', 10, 10.5, '2166-03-30', 'Active', 12000000.00),
  (8, 'LunarCruiser-4','lunar', 10, 10.5, '2166-04-15', 'Active', 12000000.00),
  (9, 'InterplanetaryArk-1','interplanetar', 20, 15.0, '2167-01-25', 'Active', 40000000.00),
  (10, 'InterplanetaryArk-2','interplanetar', 20, 15.0, '2167-02-28', 'Active', 40000000.00),
  (11, 'InterplanetaryArk-3','interplanetar', 20, 15.0, '2167-03-15', 'Active', 40000000.00),
  (12, 'InterplanetaryArk-4','interplanetar', 20, 15.0, '2167-04-20', 'Active', 40000000.00);
"""

for statement in schema_sql.strip().split(';'):
    if statement.strip():
        cursor.execute(statement)
con.commit()
print("Dane do tabel Rockets i Spaceships zostały zaktualizowane.")

Dane do tabel Rockets i Spaceships zostały zaktualizowane.


### Dodanie pracowników

In [47]:
import pandas as pd
import numpy as np
import mysql.connector
from mysql.connector import Error
from datetime import datetime, timedelta
import random

# Funkcje do generowania PESEL i daty urodzenia
def losowa_data_urodzenia():
    start = datetime(2117, 1, 1)  # Maksymalny wiek 50 lat w 2167
    end = datetime(2146, 12, 31)  # Minimalny wiek 21 lat w 2167
    delta = end - start
    random_days = random.randint(0, delta.days)
    return (start + timedelta(days=random_days)).date()

def generuj_pesel(data_urodzenia, gender):
    rok = data_urodzenia.year
    miesiac = data_urodzenia.month
    dzien = data_urodzenia.day
    
    # Kodowanie miesiąca dla lat 2100–2199
    if 2100 <= rok <= 2199:
        miesiac += 40
    
    # Cyfra płci: nieparzysta dla mężczyzn, parzysta dla kobiet
    if gender == 'M':
        cyfra_plci = random.choice([1, 3, 5, 7, 9])
    else:
        cyfra_plci = random.choice([0, 2, 4, 6, 8])
    
    losowa_czesc = f"{random.randint(0, 999):03d}{cyfra_plci}"
    pesel_bez_kontrolnej = f"{rok % 100:02d}{miesiac:02d}{dzien:02d}{losowa_czesc}"
    
    wagi = [1, 3, 7, 9, 1, 3, 7, 9, 1, 3]
    suma = sum(int(pesel_bez_kontrolnej[i]) * wagi[i] for i in range(10))
    cyfra_kontrolna = (10 - (suma % 10)) % 10
    
    return f"{pesel_bez_kontrolnej}{cyfra_kontrolna}"

# Wczytanie danych z plików CSV
imiona_men_df = pd.read_csv('../data/imiona_men.csv')
nazwiska_men_df = pd.read_csv('../data/nazwiska_men.csv')
imiona_woman_df = pd.read_csv('../data/imiona_woman.csv')
nazwiska_woman_df = pd.read_csv('../data/nazwiska_woman.csv')

# Zakładam, że pliki CSV mają kolumny 'imie' i 'nazwisko'
men_names = imiona_men_df['imie'].tolist()
men_surnames = nazwiska_men_df['nazwisko'].tolist()
women_names = imiona_woman_df['imie'].tolist()
women_surnames = nazwiska_woman_df['nazwisko'].tolist()

# Struktura korporacji
employees_structure = [
    {'role': 'CEO', 'specialization': 'executive', 'count': 1, 'salary': 5000000.00},
    {'role': 'department_head_service', 'specialization': 'service_management', 'count': 1, 'salary': 500000.00},
    {'role': 'department_head_pilot', 'specialization': 'pilot_management', 'count': 1, 'salary': 500000.00},
    {'role': 'department_head_mechanic', 'specialization': 'mechanic_management', 'count': 1, 'salary': 500000.00},
    {'role': 'department_head_admin', 'specialization': 'admin_management', 'count': 1, 'salary': 500000.00},
    {'role': 'service_staff', 'specialization': 'service', 'count': 10, 'salary': 80000.00},
    {'role': 'pilot', 'specialization': 'pilot', 'count': 100, 'salary': 150000.00},
    {'role': 'mechanic', 'specialization': 'rockets', 'count': 50, 'salary': 100000.00},
    {'role': 'mechanic', 'specialization': 'spaceships', 'count': 50, 'salary': 100000.00},
    {'role': 'admin', 'specialization': 'administration', 'count': 4, 'salary': 70000.00},
    {'role': 'space_airport_staff', 'specialization': 'airport_operations', 'count': 100, 'salary': 50000.00}
]

# Generowanie danych pracowników
employees = []
used_pesels = set()  # Zbiór do śledzenia unikalnych numerów PESEL

for emp in employees_structure:
    role = emp['role']
    specialization = emp['specialization']
    count = emp['count']
    salary = emp['salary']
    
    for _ in range(count):
        # Losowanie płci (50% mężczyzn, 50% kobiet)
        gender = np.random.choice(['M', 'F'])
        if gender == 'M':
            first_name = np.random.choice(men_names)
            last_name = np.random.choice(men_surnames)
        else:
            first_name = np.random.choice(women_names)
            last_name = np.random.choice(women_surnames)
        
        # Generowanie daty urodzenia i unikalnego PESEL
        while True:
            date_of_birth = losowa_data_urodzenia()
            pesel = generuj_pesel(date_of_birth, gender)
            if pesel not in used_pesels:
                used_pesels.add(pesel)
                break
        
        employees.append({
            'pesel': pesel,
            'first_name': first_name,
            'last_name': last_name,
            'sex': gender,
            'date_of_birth': date_of_birth,
            'role': role,
            'specialization': specialization,
            'salary': salary
        })

# Połączenie z bazą danych i wstawianie danych
try:
    con = mysql.connector.connect(
        host = "giniewicz.it",
        user = "team07",
        password = "te@mlot",
        database = "team07"  
    )
    if con.is_connected():
        cursor = con.cursor()
        
        # Usunięcie istniejących rekordów z tabeli Employees
        cursor.execute("DELETE FROM Employees;")
        
        # Definicja rozmiaru partii
        batch_size = 50
        
        # Wstawianie rekordów w partiach za pomocą executemany
        for i in range(0, len(employees), batch_size):
            batch = employees[i:i + batch_size]
            # Przygotowanie danych jako listy krotek
            data = [
                (
                    emp['pesel'],
                    emp['first_name'],
                    emp['last_name'],
                    emp['sex'],
                    emp['date_of_birth'],
                    emp['role'],
                    emp['specialization'],
                    emp['salary']
                ) for emp in batch
            ]
            try:
                # Wykonanie zapytania parametryzowanego z poprawnymi placeholderami
                cursor.executemany(
                    "INSERT INTO Employees (pesel, first_name, last_name, sex, date_of_birth, role, specialization, salary) VALUES (%s, %s, %s, %s, %s, %s, %s, %s)",
                    data
                )
                print(f"Inserted batch {i // batch_size + 1} with {len(batch)} records.")
            except Error as e:
                print(f"Błąd podczas wstawiania partii {i // batch_size + 1}: {e}")
        
        # Zatwierdzenie zmian w bazie danych
        con.commit()
        print("Dane do tabeli Employees zostały wstawione.")
        
except Error as e:
    print(f"Błąd połączenia z bazą danych: {e}")
finally:
    if con.is_connected():
        cursor.close()
        con.close()
        print("Połączenie z bazą danych zostało zamknięte.")

Inserted batch 1 with 50 records.
Inserted batch 2 with 50 records.
Inserted batch 3 with 50 records.
Inserted batch 4 with 50 records.
Inserted batch 5 with 50 records.
Inserted batch 6 with 50 records.
Inserted batch 7 with 19 records.
Dane do tabeli Employees zostały wstawione.
Połączenie z bazą danych zostało zamknięte.


In [7]:

con = mysql.connector.connect(
    host = "giniewicz.it",
    user = "team07",
    password = "te@mlot",
    database = "team07"
)

if(con):
    print("Połączenie udane")
else:
    print("Połączenie nieudane")

Połączenie udane


## SYMULACJA TEST

In [59]:
import random
import mysql.connector
from datetime import datetime, timedelta
from solarsystemtimeanddistcalc import calculate_trip_times
import numpy as np
import pandas as pd

# Funkcje generujące dane klientów
def losowa_data_urodzenia():
    start = datetime(2080, 1, 1)
    end = datetime(2149, 12, 31)
    delta = end - start
    random_days = random.randint(0, delta.days)
    return (start + timedelta(days=random_days)).date()

def generuj_pesel(data_urodzenia, sex):
    rok = data_urodzenia.year
    miesiac = data_urodzenia.month
    dzien = data_urodzenia.day
    
    if 2100 <= rok <= 2199:
        miesiac += 40
    elif 2000 <= rok <= 2099:
        miesiac += 20
    elif 2200 <= rok <= 2299:
        miesiac += 60
    
    if sex == 'M':
        cyfra_plci = random.choice([1, 3, 5, 7, 9])
    else:
        cyfra_plci = random.choice([0, 2, 4, 6, 8])
    
    losowa_czesc = f"{random.randint(0, 999):03d}{cyfra_plci}"
    pesel_bez_kontrolnej = f"{rok % 100:02d}{miesiac:02d}{dzien:02d}{losowa_czesc}"
    
    wagi = [1, 3, 7, 9, 1, 3, 7, 9, 1, 3]
    suma = sum(int(pesel_bez_kontrolnej[i]) * wagi[i] for i in range(10))
    cyfra_kontrolna = (10 - (suma % 10)) % 10
    
    return f"{pesel_bez_kontrolnej}{cyfra_kontrolna}"

def wczytaj_dane_imion_nazwisk():
    imiona_men_df = pd.read_csv('../data/imiona_men.csv')
    nazwiska_men_df = pd.read_csv('../data/nazwiska_men.csv')
    imiona_woman_df = pd.read_csv('../data/imiona_woman.csv')
    nazwiska_woman_df = pd.read_csv('../data/nazwiska_woman.csv')

    men_names = imiona_men_df['imie'].tolist()
    men_surnames = nazwiska_men_df['nazwisko'].tolist()
    women_names = imiona_woman_df['imie'].tolist()
    women_surnames = nazwiska_woman_df['nazwisko'].tolist()
    sample_size = min(1000, len(men_names), len(men_surnames), len(women_names), len(women_surnames))
    
    sampled_imiona_men = random.sample(men_names, sample_size)
    sampled_nazwiska_men = random.sample(men_surnames, sample_size)
    sampled_imiona_woman = random.sample(women_names, sample_size)
    sampled_nazwiska_woman = random.sample(women_surnames, sample_size)

    dane = {
        'M': list(zip(sampled_imiona_men, sampled_nazwiska_men)),
        'F': list(zip(sampled_imiona_woman, sampled_nazwiska_woman))
    }
    
    return dane

def generate_customers_for_trip(cursor, num_customers, dane_imion_nazwisk):
    """Generuje klientów dla jednej wycieczki i zapisuje ich do tabeli Customers."""
    licznik_men = 0
    licznik_woman = 0
    generated_pesels = []

    for i in range(num_customers):
        sex = random.choice(['M', 'F'])
        
        if sex == 'M':
            first_name, last_name = dane_imion_nazwisk['M'][licznik_men % len(dane_imion_nazwisk['M'])]
            licznik_men += 1
        else:
            first_name, last_name = dane_imion_nazwisk['F'][licznik_woman % len(dane_imion_nazwisk['F'])]
            licznik_woman += 1
        
        date_of_birth = losowa_data_urodzenia()
        pesel = generuj_pesel(date_of_birth, sex)
        
        email = f"{first_name.lower()}.{last_name.lower()}@example.com"
        phone_number = f"+48{random.randint(100000000, 999999999)}"
        emergency_contact_name = random.choice(['Jan Kowalski', 'Anna Nowak', 'Piotr Wiśniewski', 'Maria Zielińska'])
        emergency_contact_phone = f"+48{random.randint(100000000, 999999999)}"
        emergency_contact_relation = random.choice(['spouse', 'parent', 'sibling', 'friend'])
        is_insured = random.choice([True, False])
        
        try:
            cursor.execute("""
                INSERT INTO Customers 
                (pesel, first_name, last_name, date_of_birth, sex, email, is_insured, phone_number, 
                emergency_contact_name, emergency_contact_phone, emergency_contact_relation)
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
            """, (pesel, first_name, last_name, date_of_birth, sex, email, is_insured, phone_number,
                  emergency_contact_name, emergency_contact_phone, emergency_contact_relation))
            generated_pesels.append(pesel)
        except mysql.connector.IntegrityError as e:
            print(f"Duplikat PESEL {pesel} pominięty, generowanie nowego...")
            continue

    return generated_pesels

def generate_customers(start_year, end_year, step=0.5, initial_customers=100, trend=2, noise=15):
    """Generuje liczbę klientów dla każdego półrocza w podanym okresie."""
    years = np.arange(start_year, end_year + step, step)
    n_periods = len(years)
    np.random.seed(42)
    customers = initial_customers + trend * np.arange(n_periods)
    customers = customers + np.random.normal(0, noise, n_periods)
    customers = np.round(customers).astype(int)
    customers = np.maximum(customers, 0)
    return years, customers

def get_required_vehicle_type(destination_name):
    """Zwraca wymagany typ pojazdu dla danej destynacji."""
    if destination_name in ['Mars', 'Mercury', 'Venus']:
        return 'interplanetar'
    elif destination_name == 'Moon':
        return 'lunar'
    elif destination_name == 'Space Station':
        return 'orbital'
    return None

def run_simulation(start_budget, start_date, end_date):
    try:
        con = mysql.connector.connect(
            host="giniewicz.it",
            user="team07",
            password="te@mlot",
            database="team07",
            charset="utf8"
        )
        cursor = con.cursor(dictionary=True)

        print("Czyszczenie tabel Customers, Trips, TripParticipants i RecoveryTasks...")
        cursor.execute("SET FOREIGN_KEY_CHECKS = 0")
        cursor.execute("DELETE FROM Customers")
        cursor.execute("DELETE FROM Trips")
        cursor.execute("DELETE FROM TripParticipants")
        cursor.execute("DELETE FROM RecoveryTasks")
        cursor.execute("ALTER TABLE Trips AUTO_INCREMENT = 1")
        cursor.execute("SET FOREIGN_KEY_CHECKS = 1")
        con.commit()
        print("Tabele zostały wyczyszczone, autoinkrementacja Trips zresetowana.")

        dane_imion_nazwisk = wczytaj_dane_imion_nazwisk()
        budget = start_budget
        print(f"Startowy budżet: {budget}")

        years, customers = generate_customers(start_date.year, end_date.year, step=0.5)
        customer_index = 0

        current_date = start_date
        while current_date <= end_date:
            print(f"\nSymulacja dla daty: {current_date}")

            if customer_index < len(customers):
                total_customers = customers[customer_index]
                print(f"Liczba klientów: {total_customers}")
            else:
                print("Brak danych o klientach dla tego okresu.")
                break

            while total_customers > 0:
                max_attempts = 20
                attempt = 0
                while attempt < max_attempts:
                    cursor.execute("SELECT * FROM Destinations ORDER BY RAND() LIMIT 1")
                    destination = cursor.fetchone()

                    required_vehicle_type = get_required_vehicle_type(destination['name'])
                    if not required_vehicle_type:
                        print(f"Nieznana destynacja: {destination['name']}. Pomijanie wycieczki.")
                        attempt += 1
                        continue

                    cursor.execute("""
                        SELECT spaceship_id, capacity, travel_speed, status, initial_cost 
                        FROM Spaceships 
                        WHERE status = 'active' AND type = %s
                    """, (required_vehicle_type,))
                    spaceships = cursor.fetchall()

                    launch_date = current_date + timedelta(days=random.randint(0, 181))
                    cursor.execute("""
                        SELECT r.rocket_id, r.status, r.initial_cost 
                        FROM Rockets r
                        LEFT JOIN RecoveryTasks rt ON r.rocket_id = rt.rocket_id
                        WHERE r.status = 'active' 
                        AND r.type = %s
                        AND (rt.rocket_id IS NULL OR rt.end_date <= %s OR rt.status != 'in_progress')
                    """, (required_vehicle_type, launch_date))
                    rockets = cursor.fetchall()

                    if not spaceships or not rockets:
                        print(f"Brak dostępnych statków lub rakiet typu {required_vehicle_type} dla destynacji {destination['name']} na {launch_date}.")
                        attempt += 1
                        continue
                    break

                if attempt >= max_attempts:
                    print("Nie udało się znaleźć dostępnej destynacji z odpowiednimi statkami i rakietami. Sprawdź liczbę dostępnych rakiet w bazie danych.")
                    break

                for spaceship in spaceships:
                    if total_customers <= 0:
                        break

                    launch_date = current_date + timedelta(days=random.randint(0, 181))
                    cursor.execute("""
                        SELECT r.rocket_id, r.status, r.initial_cost 
                        FROM Rockets r
                        LEFT JOIN RecoveryTasks rt ON r.rocket_id = rt.rocket_id
                        WHERE r.status = 'active' 
                        AND r.type = %s
                        AND (rt.rocket_id IS NULL OR rt.end_date <= %s OR rt.status != 'in_progress')
                    """, (required_vehicle_type, launch_date))
                    rockets = cursor.fetchall()

                    if not rockets:
                        print(f"Brak dostępnych rakiet typu {required_vehicle_type} na {launch_date}. Pomijanie wycieczki.")
                        continue

                    rocket = random.choice(rockets)
                    travel_to_days, total_time_days, travel_back_days = calculate_trip_times(
                        launch_date, 
                        random.randint(3, 14),
                        destination['destination_id'],
                        spaceship['travel_speed']
                    )

                    if not travel_to_days:
                        print(f"Nie udało się obliczyć czasu podróży dla destynacji {destination['name']} na {launch_date}. Prawdopodobnie problem z danymi destynacji.")
                        continue

                    base_cost = random.randint(100000, 500000)
                    organization_cost = (
                        (spaceship['capacity'] / max(total_customers, 1)) * sum_employee_salaries(cursor) +
                        float(rocket['initial_cost']) * 0.7 +
                        float(spaceship['initial_cost']) * 0.15 +
                        base_cost
                    )
                    budget -= organization_cost
                    if budget < 0:
                        print(f"Budżet wyczerpany: {budget}. Kończenie symulacji.")
                        con.commit()
                        return

                    return_date = launch_date + timedelta(days=total_time_days)
                    cursor.execute("""
                        INSERT INTO Trips (name, description, destination_id, launch_date, duration_of_stay, return_date, spaceship_id, rocket_id, status, organization_cost)
                        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, 'planned', %s)
                    """, (
                        f"Wycieczka do {destination['name']}",
                        destination['description'],
                        destination['destination_id'],
                        launch_date,
                        total_time_days,
                        return_date,
                        spaceship['spaceship_id'],
                        rocket['rocket_id'],
                        organization_cost
                    ))
                    trip_id = cursor.lastrowid

                    participants = min(spaceship['capacity'], total_customers)
                    generated_pesels = generate_customers_for_trip(cursor, participants, dane_imion_nazwisk)
                    actual_participants = len(generated_pesels)
                    
                    if actual_participants == 0:
                        print(f"Nie udało się wygenerować klientów dla wycieczki {trip_id}.")
                        continue

                    for pesel in generated_pesels:
                        ticket_price = organization_cost / max(actual_participants, 1)
                        ticket_price = min(round(ticket_price, 2), 999999.99)  # Ograniczenie ticket_price
                        cursor.execute("""
                            INSERT INTO TripParticipants (trip_id, pesel, ticket_price, status, satisfaction_level)
                            VALUES (%s, %s, %s, 'confirmed', %s)
                        """, (
                            trip_id,
                            pesel,
                            ticket_price,
                            round(random.uniform(0.5, 1.0), 2)
                        ))
                        total_customers -= 1

                    print(f"Zorganizowano wycieczkę {trip_id} do {destination['name']} z {actual_participants} uczestnikami na {launch_date}.")

                    repair_start_date = return_date
                    repair_end_date = repair_start_date + timedelta(days=random.randint(30, 60))
                    cursor.execute("""
                        INSERT INTO RecoveryTasks (pesel, rocket_id, start_date, end_date, status)
                        VALUES (%s, %s, %s, %s, 'in_progress')
                    """, (
                        assign_employee(cursor, 'rockets'),
                        rocket['rocket_id'],
                        repair_start_date,
                        repair_end_date
                    ))

            if current_date.month == 12:
                salaries = sum_employee_salaries(cursor)
                budget -= salaries
                if budget < 0:
                    print(f"Budżet wyczerpany po wypłacie pensji: {budget}. Kończenie symulacji.")
                    con.commit()
                    return
                print(f"Wypłacono pensje pracownikom: {salaries}. Pozostały budżet: {budget}")

            current_date += timedelta(days=182)
            customer_index += 1

        con.commit()
        print(f"Symulacja zakończona. Pozostały budżet: {budget}")

    except mysql.connector.Error as err:
        print(f"Błąd podczas operacji na bazie danych: {err}")
    finally:
        if 'con' in locals() and con.is_connected():
            cursor.close()
            con.close()
            print("Połączenie z bazą danych zostało zamknięte.")

def sum_employee_salaries(cursor):
    """Oblicza sumę pensji wszystkich pracowników."""
    cursor.execute("SELECT SUM(salary) AS total_salaries FROM Employees")
    result = cursor.fetchone()
    total_salaries = float(result['total_salaries']) if result['total_salaries'] is not None else 0.0
    return total_salaries

def assign_employee(cursor, specialization):
    """Przypisuje pracownika o danej specjalizacji."""
    cursor.execute("SELECT pesel FROM Employees WHERE specialization = %s ORDER BY RAND() LIMIT 1", (specialization,))
    employee = cursor.fetchone()
    return employee['pesel'] if employee else None

if __name__ == "__main__":
    start_budget = 100000000
    start_date = datetime(2167, 1, 1)
    end_date = datetime(2170, 12, 31)
    run_simulation(start_budget, start_date, end_date)

Czyszczenie tabel Customers, Trips, TripParticipants i RecoveryTasks...
Tabele zostały wyczyszczone, autoinkrementacja Trips zresetowana.
Startowy budżet: 100000000

Symulacja dla daty: 2167-01-01 00:00:00
Liczba klientów: 107
Nie udało się obliczyć czasu podróży dla destynacji Moon na 2167-02-27 00:00:00. Prawdopodobnie problem z danymi destynacji.
Nie udało się obliczyć czasu podróży dla destynacji Moon na 2167-02-01 00:00:00. Prawdopodobnie problem z danymi destynacji.
Nie udało się obliczyć czasu podróży dla destynacji Moon na 2167-01-04 00:00:00. Prawdopodobnie problem z danymi destynacji.
Nie udało się obliczyć czasu podróży dla destynacji Moon na 2167-05-27 00:00:00. Prawdopodobnie problem z danymi destynacji.


c:\Users\Admin\Space-U\scripts\solarsystemtimeanddistcalc.py:49: RuntimeWarning: The iteration is not making good progress, as measured by the 
 improvement from the last five Jacobian evaluations.
  delta_t, = fsolve(func, guess)
c:\Users\Admin\Space-U\scripts\solarsystemtimeanddistcalc.py:49: RuntimeWarning: The iteration is not making good progress, as measured by the 
 improvement from the last ten iterations.
  delta_t, = fsolve(func, guess)


Zorganizowano wycieczkę 1 do Venus z 20 uczestnikami na 2167-05-12 00:00:00.
Zorganizowano wycieczkę 2 do Venus z 20 uczestnikami na 2167-01-27 00:00:00.
Budżet wyczerpany: -51366722.17695602. Kończenie symulacji.
Połączenie z bazą danych zostało zamknięte.


## WYGENEROWAĆ KLIENTÓW DO WYBRANYCH TRIPÓW

## WYGENEROWANIE uzupełnienia TRIPS



In [9]:
import numpy as np
import random
from datetime import datetime, timedelta
import mysql.connector
import re

# Stała macierz ofert wycieczek
TRIP_OFFERS = [
    (4, '1-dniowy spacer po Księżycu', 'Krótka wycieczka z spacerem po powierzchni Księżyca', 1),
    (4, '3-dniowa eksploracja Księżyca', 'Wycieczka z eksploracją kraterów i baz księżycowych', 3),
    (3, 'Dwutygodniowy obóz na Marsie', 'Obóz survivalowy z eksploracją Czerwonej Planety', 14),
    (3, '5-dniowa misja na Marsie', 'Krótka misja z wizytą w bazie mariańskiej', 5),
    (2, 'Tygodniowa przygoda na Wenus', 'Wycieczka z ochroną przed wysokimi temperaturami', 7),
    (2, '3-dniowy przelot nad Wenus', 'Podniebna wycieczka z widokiem na atmosferę Wenus', 3),
    (5, '2-dniowy pobyt na Stacji Kosmicznej', 'Odwiedziny na orbicie z treningiem astronautycznym', 2),
    (5, '4-dniowy wypoczynek na Stacji Kosmicznej', 'Relaks i obserwacje Ziemi z orbity', 4),
    (1, '10-dniowa ekspedycja na Merkurego', 'Wycieczka zbadaj najgorętszą planetę Układu Słonecznego', 10),
    (1, '6-dniowa misja na Merkurego', 'Krótka misja z obserwacjami powierzchni Merkurego', 6),
    (4, 'Weekend na Księżycu', '2-dniowa wycieczka z noclegiem na Księżycu', 2),
    (3, 'Miesiąc na Marsie', 'Długoterminowy pobyt z naukowymi eksperymentami', 30),
    (2, '8-dniowa wyprawa na Wenus', 'Wycieczka z badaniem geologii Wenus', 8),
    (5, '3-dniowy kurs astronautyczny', 'Szkolenie na Stacji Kosmicznej', 3),
    (1, '12-dniowa eksploracja Merkurego', 'Rozszerzona misja z analizą kraterów', 12)
]

# Połączenie z bazą danych
con = mysql.connector.connect(
    host="giniewicz.it",
    user="team07",
    password="te@mlot",
    database="team07",
    charset="utf8"
)
cursor = con.cursor(dictionary=True)

# Funkcja sprawdzająca dostępność zasobów z uwzględnieniem typu, travel_speed i 30-dniowego okresu przestoju
def get_available_resources(launch_date, destination_id):
    one_month_ago = launch_date - timedelta(days=30)
    downtime_period = launch_date - timedelta(days=30)  # Minimum 30 dni od ostatniego powrotu
    
    # Mapowanie destination_id do typu
    type_mapping = {
        1: 'interplanetar',  # Mercury
        2: 'interplanetar',  # Venus
        3: 'interplanetar',  # Mars
        4: 'lunar',          # Moon
        5: 'orbital'         # Space Station
    }
    required_type = type_mapping.get(destination_id, 'interplanetar')
    
    # Pobierz dostępne spaceships z travel_speed, uwzględniając 30-dniowy przestój
    cursor.execute("""
        SELECT s.spaceship_id, s.capacity, s.travel_speed 
        FROM Spaceships s
        LEFT JOIN (
            SELECT spaceship_id, MAX(return_date) as last_return
            FROM Trips
            GROUP BY spaceship_id
        ) t ON s.spaceship_id = t.spaceship_id
        WHERE t.last_return IS NULL OR t.last_return <= %s
        AND s.type = %s
    """, (downtime_period, required_type))
    spaceships = cursor.fetchall()
    
    # Pobierz dostępne rockets, uwzględniając 30-dniowy przestój
    cursor.execute("""
        SELECT r.rocket_id 
        FROM Rockets r
        LEFT JOIN (
            SELECT rocket_id, MAX(return_date) as last_return
            FROM Trips
            GROUP BY rocket_id
        ) t ON r.rocket_id = t.rocket_id
        WHERE t.last_return IS NULL OR t.last_return <= %s
        AND r.type = %s
    """, (downtime_period, required_type))
    rockets = cursor.fetchall()
    
    return spaceships, rockets, required_type

# Generowanie wycieczek
def generate_trips():
    # Czyszczenie tabeli Trips przed generowaniem (usunięcie wszystkich rekordów i zresetowanie auto_increment)
    cursor.execute("SET FOREIGN_KEY_CHECKS = 0;")
    cursor.execute("DELETE FROM Trips;")
    cursor.execute("ALTER TABLE Trips AUTO_INCREMENT = 1;")
    cursor.execute("SET FOREIGN_KEY_CHECKS = 1;")
    con.commit()
    print("Tabela Trips została wyczyszczona (DELETE) i trip_id zresetowany do 1.")

    years = np.arange(2168, 2188, 24/12)  # Krok co pół roku
    base_customers = 100
    growth = 4
    noise = 15
    
    np.random.seed(42)
    customers = base_customers + growth * np.arange(len(years))
    customers = customers + np.random.normal(0, noise, len(years))
    customers = np.round(customers).astype(int)
    
    for i, (year, customer_count) in enumerate(zip(years, customers)):
        print(f"\nProcessing {year:.1f} - {customer_count} customers")
        
        # Przywrócenie listy TRIP_OFFERS na początku każdego roku
        if i > 0:
            TRIP_OFFERS[:] = [
                (4, '1-dniowy spacer po Księżycu', 'Krótka wycieczka z spacerem po powierzchni Księżyca', 1),
                (4, '3-dniowa eksploracja Księżyca', 'Wycieczka z eksploracją kraterów i baz księżycowych', 3),
                (3, 'Dwutygodniowy obóz na Marsie', 'Obóz survivalowy z eksploracją Czerwonej Planety', 14),
                (3, '5-dniowa misja na Marsie', 'Krótka misja z wizytą w bazie mariańskiej', 5),
                (2, 'Tygodniowa przygoda na Wenus', 'Wycieczka z ochroną przed wysokimi temperaturami', 7),
                (2, '3-dniowy przelot nad Wenus', 'Podniebna wycieczka z widokiem na atmosferę Wenus', 3),
                (5, '2-dniowy pobyt na Stacji Kosmicznej', 'Odwiedziny na orbicie z treningiem astronautycznym', 2),
                (5, '4-dniowy wypoczynek na Stacji Kosmicznej', 'Relaks i obserwacje Ziemi z orbity', 4),
                (1, '10-dniowa ekspedycja na Merkurego', 'Wycieczka zbadaj najgorętszą planetę Układu Słonecznego', 10),
                (1, '6-dniowa misja na Merkurego', 'Krótka misja z obserwacjami powierzchni Merkurego', 6),
                (4, 'Weekend na Księżycu', '2-dniowa wycieczka z noclegiem na Księżycu', 2),
                (3, 'Miesiąc na Marsie', 'Długoterminowy pobyt z naukowymi eksperymentami', 30),
                (2, '8-dniowa wyprawa na Wenus', 'Wycieczka z badaniem geologii Wenus', 8),
                (5, '3-dniowy kurs astronautyczny', 'Szkolenie na Stacji Kosmicznej', 3),
                (1, '12-dniowa eksploracja Merkurego', 'Rozszerzona misja z analizą kraterów', 12)
            ]
        print(f"Remaining TRIP_OFFERS: {len(TRIP_OFFERS)}")
        
        remaining_customers = customer_count
        trip_date = datetime(int(year), 6 if (year % 1) > 0.5 else 1, 1)
        
        while remaining_customers > 0 and TRIP_OFFERS:
            # Wybierz losową ofertę wycieczki
            offer = random.choice(TRIP_OFFERS)
            destination_id, name, description, duration = offer
            
            # Pobierz dostępne zasoby dla tej daty i typu destynacji
            spaceships, rockets, required_type = get_available_resources(trip_date, destination_id)
            
            if not spaceships or not rockets:
                TRIP_OFFERS.remove(offer)
                continue
            
            # Znajdź kompatybilny statek dla tej destynacji (z travel_speed)
            cursor.execute("""
                SELECT s.spaceship_id, s.capacity, s.travel_speed 
                FROM Spaceships s
                WHERE s.spaceship_id IN ({})
                AND s.type = %s
                ORDER BY RAND() LIMIT 1
            """.format(', '.join(['%s'] * len(spaceships))), 
            [s['spaceship_id'] for s in spaceships] + [required_type])
            spaceship = cursor.fetchone()
            
            if not spaceship:
                TRIP_OFFERS.remove(offer)
                continue
            
            # Wybierz losową rakietę
            rocket = random.choice(rockets)
            
            # Pobierz distance_range dla danej destynacji
            cursor.execute("SELECT distance_range FROM Destinations WHERE destination_id = %s", (destination_id,))
            destination = cursor.fetchone()
            if not destination or not destination['distance_range']:
                distance = 0  # Default if no distance
            else:
                # Wyodrębnij liczbę z ciągu (np. "0.00000267 AU" -> 0.00000267)
                distance_str = re.search(r'[-+]?\d*\.\d+|\d+', destination['distance_range']).group()
                distance = float(distance_str)
            
            # Oblicz czas podróży (distance / travel_speed w dniach, zakładając travel_speed w km/s i distance w AU)
            travel_speed = spaceship['travel_speed']  # w km/s
            distance_km = distance * 149600000  # Konwersja AU na km
            travel_time_days = (distance_km / (travel_speed * 86400))  # Dni (86400 s = 1 dzień)
            travel_time_days = max(1, travel_time_days)  # Minimum 1 dzień
            
            # Oblicz liczbę uczestników (nie więcej niż pojemność statku)
            participants = min(spaceship['capacity'], remaining_customers)
            
            # Data powrotu (launch_date + travel_time + duration + travel_time)
            return_date = trip_date + timedelta(days=travel_time_days) + timedelta(days=duration) + timedelta(days=travel_time_days)
            
            # Wstaw wycieczkę do bazy (bez trip_id)
            cursor.execute("""
                INSERT INTO Trips (
                    destination_id, 
                    name,
                    description,
                    launch_date, 
                    return_date, 
                    spaceship_id,
                    duration_of_stay,
                    rocket_id,
                    status
                ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, 'planned')
            """, (
                destination_id,
                name,
                description,
                trip_date,
                return_date,
                spaceship['spaceship_id'],
                duration,
                rocket['rocket_id']
            ))
            trip_id = cursor.lastrowid
            
            print(f"Created trip {trip_id}: {name} for {participants} customers (Duration: {duration} days, Travel Time: {travel_time_days:.1f} days each way, Return Date: {return_date})")
            
            remaining_customers -= participants
            trip_date += timedelta(days=30)  # Następna wycieczka za miesiąc
    
    con.commit()

generate_trips()
cursor.close()
con.close()

Tabela Trips została wyczyszczona (DELETE) i trip_id zresetowany do 1.

Processing 2168.0 - 107 customers
Remaining TRIP_OFFERS: 15
Created trip 1: 5-dniowa misja na Marsie for 20 customers (Duration: 5 days, Travel Time: 60.0 days each way, Return Date: 2168-05-05 01:11:06.666666)
Created trip 2: 2-dniowy pobyt na Stacji Kosmicznej for 5 customers (Duration: 2 days, Travel Time: 1.0 days each way, Return Date: 2168-02-04 00:00:00)
Created trip 3: Miesiąc na Marsie for 20 customers (Duration: 30 days, Travel Time: 60.0 days each way, Return Date: 2168-07-29 01:11:06.666666)
Created trip 4: 3-dniowy przelot nad Wenus for 20 customers (Duration: 3 days, Travel Time: 31.2 days each way, Return Date: 2168-06-04 08:00:00)
Created trip 5: Tygodniowa przygoda na Wenus for 20 customers (Duration: 7 days, Travel Time: 31.2 days each way, Return Date: 2168-07-08 08:00:00)
Created trip 6: 3-dniowy kurs astronautyczny for 5 customers (Duration: 3 days, Travel Time: 1.0 days each way, Return Date: 

In [10]:
import random
from datetime import datetime, timedelta
import mysql.connector
import pandas as pd

# Funkcje generujące dane
def losowa_data_urodzenia():
    start = datetime(2080, 1, 1)
    end = datetime(2149, 12, 31)
    delta = end - start
    random_days = random.randint(0, delta.days)
    return (start + timedelta(days=random_days)).date()

def generuj_pesel(data_urodzenia, sex):
    rok = data_urodzenia.year
    miesiac = data_urodzenia.month
    dzien = data_urodzenia.day
    
    if 2100 <= rok <= 2199:
        miesiac += 40
    elif 2000 <= rok <= 2099:
        miesiac += 20
    elif 2200 <= rok <= 2299:
        miesiac += 60
    
    if sex == 'M':
        cyfra_plci = random.choice([1, 3, 5, 7, 9])
    else:
        cyfra_plci = random.choice([0, 2, 4, 6, 8])
    
    losowa_czesc = f"{random.randint(0, 999):03d}{cyfra_plci}"
    pesel_bez_kontrolnej = f"{rok % 100:02d}{miesiac:02d}{dzien:02d}{losowa_czesc}"
    
    wagi = [1, 3, 7, 9, 1, 3, 7, 9, 1, 3]
    suma = sum(int(pesel_bez_kontrolnej[i]) * wagi[i] for i in range(10))
    cyfra_kontrolna = (10 - (suma % 10)) % 10
    
    return f"{pesel_bez_kontrolnej}{cyfra_kontrolna}"

def wczytaj_dane_imion_nazwisk():
    imiona_men_df = pd.read_csv('../data/imiona_men.csv')
    nazwiska_men_df = pd.read_csv('../data/nazwiska_men.csv')
    imiona_woman_df = pd.read_csv('../data/imiona_woman.csv')
    nazwiska_woman_df = pd.read_csv('../data/nazwiska_woman.csv')

    men_names = imiona_men_df['imie'].tolist()
    men_surnames = nazwiska_men_df['nazwisko'].tolist()
    women_names = imiona_woman_df['imie'].tolist()
    women_surnames = nazwiska_woman_df['nazwisko'].tolist()
    sample_size = min(1000, len(men_names), len(men_surnames), len(women_names), len(women_surnames))
    
    sampled_imiona_men = random.sample(men_names, sample_size)
    sampled_nazwiska_men = random.sample(men_surnames, sample_size)
    sampled_imiona_woman = random.sample(women_names, sample_size)
    sampled_nazwiska_woman = random.sample(women_surnames, sample_size)

    dane = {
        'M': list(zip(sampled_imiona_men, sampled_nazwiska_men)),
        'F': list(zip(sampled_imiona_woman, sampled_nazwiska_woman))
    }
    
    return dane

def generate_customers():
    # Wczytanie danych imion i nazwisk
    dane_imion_nazwisk = wczytaj_dane_imion_nazwisk()
    
    try:
        con = mysql.connector.connect(
            host="giniewicz.it",
            user="team07",
            password="te@mlot",
            database="team07",
            charset="utf8"
        )
        cursor = con.cursor()

        print("Rozpoczęcie generowania danych klientów...")
        
        licznik_men = 0
        licznik_woman = 0
        
        # Opcjonalne wyczyszczenie tabeli Customers przed generowaniem (odkomentuj, jeśli chcesz reset)
        cursor.execute("SET FOREIGN_KEY_CHECKS = 0")
        cursor.execute("DELETE FROM Customers")
        con.commit()

        for i in range(1, 201):  # Poprawiono na 500, zgodnie z printem
            sex = random.choice(['M', 'F'])
            
            # Wybór imienia i nazwiska odpowiedniego do płci
            if sex == 'M':
                first_name, last_name = dane_imion_nazwisk['M'][licznik_men % len(dane_imion_nazwisk['M'])]
                licznik_men += 1
            else:
                first_name, last_name = dane_imion_nazwisk['F'][licznik_woman % len(dane_imion_nazwisk['F'])]
                licznik_woman += 1
            
            date_of_birth = losowa_data_urodzenia()
            pesel = generuj_pesel(date_of_birth, sex)
            
            try:
                cursor.execute("""
                    INSERT INTO Customers 
                    (pesel, first_name, last_name, date_of_birth, sex)
                    VALUES (%s, %s, %s, %s, %s)
                """, (pesel, first_name, last_name, date_of_birth, sex))
            except mysql.connector.IntegrityError as e:
                print(f"Duplikat PESEL {pesel} pominięty, generowanie nowego...")
                continue  # Pomiń duplikat i spróbuj ponownie w następnej iteracji
            
            if i % 100 == 0:
                print(f"Wygenerowano {i}/500 rekordów...")
        
        con.commit()
        print("Pomyślnie dodano 500 rekordów do bazy danych.")
        print(f"Podsumowanie: Mężczyźni: {licznik_men}, Kobiety: {licznik_woman}")

    except mysql.connector.Error as err:
        print(f"Błąd podczas łączenia z bazą danych: {err}")
    finally:
        if 'con' in locals() and con.is_connected():
            cursor.close()
            con.close()
            print("Połączenie z bazą danych zostało zamknięte.")

if __name__ == "__main__":
    generate_customers()

Rozpoczęcie generowania danych klientów...
Wygenerowano 100/500 rekordów...
Wygenerowano 200/500 rekordów...
Pomyślnie dodano 500 rekordów do bazy danych.
Podsumowanie: Mężczyźni: 102, Kobiety: 98
Połączenie z bazą danych zostało zamknięte.


In [11]:
import random
import mysql.connector
from datetime import datetime

def assign_customers_to_trips():
    try:
        con = mysql.connector.connect(
            host="giniewicz.it",
            user="team07",
            password="te@mlot",
            database="team07",
            charset="utf8"
        )
        cursor = con.cursor(dictionary=True)

        print("Rozpoczęcie przypisywania klientów do wycieczek...")

        # Pobierz wszystkie wycieczki z pojemnością
        cursor.execute("""
            SELECT t.trip_id, t.spaceship_id, s.capacity
            FROM Trips t
            JOIN Spaceships s ON t.spaceship_id = s.spaceship_id
            ORDER BY t.trip_id
        """)
        trips = cursor.fetchall()

        # Pobierz wszystkich klientów
        cursor.execute("SELECT pesel FROM Customers")
        customers = cursor.fetchall()

        if not trips or not customers:
            print("Brak wycieczek lub klientów w bazie danych.")
            return

        # Czyszczenie istniejących przypisań i reset auto-increment
        cursor.execute("SET FOREIGN_KEY_CHECKS = 0;")
        cursor.execute("DELETE FROM TripParticipants")
        cursor.execute("ALTER TABLE TripParticipants AUTO_INCREMENT = 1")
        cursor.execute("SET FOREIGN_KEY_CHECKS = 1;")
        con.commit()

        customer_index = 0  # Indeks do śledzenia aktualnego klienta
        total_customers = len(customers)

        for trip in trips:
            trip_id = trip['trip_id']
            capacity = trip['capacity']
            participants_needed = min(capacity, total_customers - customer_index)  # Nie więcej niż dostępnych klientów

            if participants_needed <= 0:
                print(f"Brak wystarczających klientów dla trip_id {trip_id}, pomijam.")
                continue

            # Przypisz sekwencyjnie pełną pojemność wycieczki
            participants_to_assign = participants_needed  # Użyj pełnej pojemności
            
            for _ in range(participants_to_assign):
                if customer_index >= total_customers:
                    print(f"Brak pozostałych klientów dla trip_id {trip_id}, kończę przypisywanie.")
                    break
                
                customer_id = customers[customer_index]['pesel']
                satisfaction_level = round(random.uniform(0, 1), 2)
                
                cursor.execute("""
                    INSERT INTO TripParticipants (trip_id, pesel, satisfaction_level)
                    VALUES (%s, %s, %s)
                """, (trip_id, customer_id, satisfaction_level))
                
                customer_index += 1
            
            print(f"Przypisano {participants_to_assign} uczestników do trip_id {trip_id} z satysfakcją od 0 do 1.")

        con.commit()
        print(f"Pomyślnie przypisano klientów do {len(trips)} wycieczek. Użyto {customer_index} klientów.")

    except mysql.connector.Error as err:
        print(f"Błąd podczas operacji na bazie danych: {err}")
    finally:
        if 'con' in locals() and con.is_connected():
            cursor.close()
            con.close()
            print("Połączenie z bazą danych zostało zamknięte.")

if __name__ == "__main__":
    assign_customers_to_trips()

Rozpoczęcie przypisywania klientów do wycieczek...
Przypisano 20 uczestników do trip_id 1 z satysfakcją od 0 do 1.
Przypisano 5 uczestników do trip_id 2 z satysfakcją od 0 do 1.
Przypisano 20 uczestników do trip_id 3 z satysfakcją od 0 do 1.
Przypisano 20 uczestników do trip_id 4 z satysfakcją od 0 do 1.
Przypisano 20 uczestników do trip_id 5 z satysfakcją od 0 do 1.
Przypisano 5 uczestników do trip_id 6 z satysfakcją od 0 do 1.
Przypisano 20 uczestników do trip_id 7 z satysfakcją od 0 do 1.
Przypisano 10 uczestników do trip_id 8 z satysfakcją od 0 do 1.
Przypisano 20 uczestników do trip_id 9 z satysfakcją od 0 do 1.
Przypisano 5 uczestników do trip_id 10 z satysfakcją od 0 do 1.
Przypisano 20 uczestników do trip_id 11 z satysfakcją od 0 do 1.
Przypisano 10 uczestników do trip_id 12 z satysfakcją od 0 do 1.
Przypisano 20 uczestników do trip_id 13 z satysfakcją od 0 do 1.
Przypisano 5 uczestników do trip_id 14 z satysfakcją od 0 do 1.
Brak wystarczających klientów dla trip_id 15, pomija

In [12]:
import random
import mysql.connector
from datetime import datetime, timedelta
import pandas as pd

# Funkcje generujące dane
def losowa_data_urodzenia():
    start = datetime(2080, 1, 1)
    end = datetime(2149, 12, 31)
    delta = end - start
    random_days = random.randint(0, delta.days)
    return (start + timedelta(days=random_days)).date()

def generuj_pesel(data_urodzenia, sex):
    rok = data_urodzenia.year
    miesiac = data_urodzenia.month
    dzien = data_urodzenia.day
    
    if 2100 <= rok <= 2199:
        miesiac += 40
    elif 2000 <= rok <= 2099:
        miesiac += 20
    elif 2200 <= rok <= 2299:
        miesiac += 60
    
    if sex == 'M':
        cyfra_plci = random.choice([1, 3, 5, 7, 9])
    else:
        cyfra_plci = random.choice([0, 2, 4, 6, 8])
    
    losowa_czesc = f"{random.randint(0, 999):03d}{cyfra_plci}"
    pesel_bez_kontrolnej = f"{rok % 100:02d}{miesiac:02d}{dzien:02d}{losowa_czesc}"
    
    wagi = [1, 3, 7, 9, 1, 3, 7, 9, 1, 3]
    suma = sum(int(pesel_bez_kontrolnej[i]) * wagi[i] for i in range(10))
    cyfra_kontrolna = (10 - (suma % 10)) % 10
    
    return f"{pesel_bez_kontrolnej}{cyfra_kontrolna}"

def wczytaj_dane_imion_nazwisk():
    imiona_men_df = pd.read_csv('../data/imiona_men.csv')
    nazwiska_men_df = pd.read_csv('../data/nazwiska_men.csv')
    imiona_woman_df = pd.read_csv('../data/imiona_woman.csv')
    nazwiska_woman_df = pd.read_csv('../data/nazwiska_woman.csv')

    men_names = imiona_men_df['imie'].tolist()
    men_surnames = nazwiska_men_df['nazwisko'].tolist()
    women_names = imiona_woman_df['imie'].tolist()
    women_surnames = nazwiska_woman_df['nazwisko'].tolist()
    sample_size = min(1000, len(men_names), len(men_surnames), len(women_names), len(women_surnames))
    
    sampled_imiona_men = random.sample(men_names, sample_size)
    sampled_nazwiska_men = random.sample(men_surnames, sample_size)
    sampled_imiona_woman = random.sample(women_names, sample_size)
    sampled_nazwiska_woman = random.sample(women_surnames, sample_size)

    dane = {
        'M': list(zip(sampled_imiona_men, sampled_nazwiska_men)),
        'F': list(zip(sampled_imiona_woman, sampled_nazwiska_woman))
    }
    
    return dane

def generate_and_assign_customers():
    try:
        con = mysql.connector.connect(
            host="giniewicz.it",
            user="team07",
            password="te@mlot",
            database="team07",
            charset="utf8"
        )
        cursor = con.cursor(dictionary=True)

        print("Rozpoczęcie generowania i przypisywania klientów...")

        # Pobierz wszystkie wycieczki z pojemnością (tylko te jeszcze nieprzypisane)
        cursor.execute("""
            SELECT t.trip_id, t.spaceship_id, s.capacity
            FROM Trips t
            JOIN Spaceships s ON t.spaceship_id = s.spaceship_id
            LEFT JOIN TripParticipants tp ON t.trip_id = tp.trip_id
            WHERE tp.trip_id IS NULL
            ORDER BY t.trip_id
        """)
        trips = cursor.fetchall()

        if not trips:
            print("Brak niezapisanym wycieczek.")
            return

        # Czyszczenie istniejących przypisań (opcjonalne, odkomentuj jeśli chcesz reset)
        # cursor.execute("DELETE FROM TripParticipants")
        # con.commit()

        customer_index = 0
        total_new_customers = 0
        licznik_men = 0
        licznik_woman = 0
        dane_imion_nazwisk = wczytaj_dane_imion_nazwisk()

        for batch_start in range(0, 500, 200):  # Przetwarzaj w blokach po 200
            batch_end = min(batch_start + 200, 500)
            for i in range(batch_start, batch_end):
                if random.random() < 0.95:  # 95% szans na nowego klienta
                    sex = random.choice(['M', 'F'])
                    if sex == 'M':
                        first_name, last_name = dane_imion_nazwisk['M'][licznik_men % len(dane_imion_nazwisk['M'])]
                        licznik_men += 1
                    else:
                        first_name, last_name = dane_imion_nazwisk['F'][licznik_woman % len(dane_imion_nazwisk['F'])]
                        licznik_woman += 1
                    
                    date_of_birth = losowa_data_urodzenia()
                    pesel = generuj_pesel(date_of_birth, sex)
                    
                    try:
                        cursor.execute("""
                            INSERT INTO Customers (pesel, first_name, last_name, date_of_birth, sex)
                            VALUES (%s, %s, %s, %s, %s)
                        """, (pesel, first_name, last_name, date_of_birth, sex))
                        total_new_customers += 1
                    except mysql.connector.IntegrityError:
                        print(f"Duplikat PESEL pominięty dla indeksu {i}.")
                        continue
                else:  # 5% szans na istniejącego zadowolonego klienta
                    cursor.execute("""
                        SELECT tp.pesel
                        FROM TripParticipants tp
                        JOIN Customers c ON tp.pesel = c.pesel
                        WHERE tp.satisfaction_level > 0.8
                        ORDER BY RAND()
                        LIMIT 1
                    """)
                    satisfied_customer = cursor.fetchone()
                    if satisfied_customer:
                        pesel = satisfied_customer['pesel']
                        if trips:
                            next_trip = trips.pop(0)  # Weź pierwszą niezapisaną wycieczkę
                            satisfaction_level = round(random.uniform(0, 1), 2)
                            cursor.execute("""
                                INSERT INTO TripParticipants (trip_id, pesel, satisfaction_level)
                                VALUES (%s, %s, %s)
                            """, (next_trip['trip_id'], pesel, satisfaction_level))
                            print(f"Przypisano zadowolonego klienta {pesel} do trip_id {next_trip['trip_id']}.")
                    else:
                        print("Brak zadowolonego klienta w Customers do przypisania.")

                if i % 100 == 0:
                    print(f"Wygenerowano/przypisano {i}/500 rekordów...")

            # Po każdym bloku 200, przypisz nowych klientów do wycieczek
            cursor.execute("SELECT pesel FROM Customers ORDER BY pesel DESC LIMIT %s", (total_new_customers,))
            new_customers = cursor.fetchall()
            for trip in trips[:]:  # Kopia listy, bo modyfikujemy w pętli
                if not new_customers:
                    break
                trip_id = trip['trip_id']
                capacity = trip['capacity']
                participants_needed = min(capacity, len(new_customers))
                participants_to_assign = participants_needed  # Użyj pełnej pojemności
                
                for _ in range(participants_to_assign):
                    if not new_customers:
                        break
                    pesel = new_customers.pop(0)['pesel']
                    satisfaction_level = round(random.uniform(0, 1), 2)
                    cursor.execute("""
                        INSERT INTO TripParticipants (trip_id, pesel, satisfaction_level)
                        VALUES (%s, %s, %s)
                    """, (trip_id, pesel, satisfaction_level))
                    customer_index += 1
                
                print(f"Przypisano {participants_to_assign} nowych uczestników do trip_id {trip_id}.")
                if participants_to_assign == 0:
                    trips.remove(trip)

        con.commit()
        print(f"Pomyślnie dodano {total_new_customers} nowych klientów i przypisano ich do wycieczek.")

    except mysql.connector.Error as err:
        print(f"Błąd podczas operacji na bazie danych: {err}")
    finally:
        if 'con' in locals() and con.is_connected():
            cursor.close()
            con.close()
            print("Połączenie z bazą danych zostało zamknięte.")

if __name__ == "__main__":
    generate_and_assign_customers()

Rozpoczęcie generowania i przypisywania klientów...
Wygenerowano/przypisano 0/500 rekordów...
Przypisano zadowolonego klienta 87261894191 do trip_id 15.
Przypisano zadowolonego klienta 90290540713 do trip_id 16.
Przypisano zadowolonego klienta 28480159074 do trip_id 17.
Przypisano zadowolonego klienta 82233022420 do trip_id 18.
Przypisano zadowolonego klienta 31502999624 do trip_id 19.
Przypisano zadowolonego klienta 92292993483 do trip_id 20.
Przypisano zadowolonego klienta 02523044486 do trip_id 21.
Przypisano zadowolonego klienta 99212338052 do trip_id 22.
Wygenerowano/przypisano 100/500 rekordów...
Przypisano zadowolonego klienta 42431439192 do trip_id 23.
Przypisano zadowolonego klienta 13442570970 do trip_id 24.
Przypisano zadowolonego klienta 45521167757 do trip_id 25.
Przypisano zadowolonego klienta 29431634024 do trip_id 26.
Przypisano 20 nowych uczestników do trip_id 27.
Przypisano 20 nowych uczestników do trip_id 28.
Przypisano 20 nowych uczestników do trip_id 29.
Przypisano

In [13]:
import random
import mysql.connector
from datetime import datetime
import pandas as pd

try:
    con = mysql.connector.connect(
        host="giniewicz.it",
        user="team07",
        password="te@mlot",
        database="team07",
        charset="utf8"
    )
    cursor = con.cursor()

    # 1. Dodawanie emaili
    print("Rozpoczęcie aktualizacji emaili...")
    cursor.execute("SELECT pesel, first_name, last_name FROM Customers")
    klienci = cursor.fetchall()

    dane_do_emaila = []
    for pesel, first, last in klienci:
        email = f"{first.lower()}_{last.lower()}@email.com".replace(" ", "").replace("ł", "l").replace("ń", "n")
        dane_do_emaila.append((email, pesel))
    sql = "UPDATE Customers SET email = %s WHERE pesel = %s"
    cursor.executemany(sql, dane_do_emaila)
    con.commit()
    print(f"Zaktualizowano emaile dla {len(dane_do_emaila)} klientów.")

    # 2. Dodawanie statusu ubezpieczenia
    print("Rozpoczęcie aktualizacji kolumny is_insured...")
    def losowe_ubezpieczenie():
        return random.choices([True, False], weights=[95, 5], k=1)[0]
    cursor.execute("SELECT pesel FROM Customers")
    ids = cursor.fetchall()
    dane_ubezpieczenia = [(losowe_ubezpieczenie(), id_[0]) for id_ in ids]
    sql = "UPDATE Customers SET is_insured = %s WHERE pesel = %s"
    cursor.executemany(sql, dane_ubezpieczenia)
    con.commit()
    print(f"Zaktualizowano kolumnę is_insured dla {len(dane_ubezpieczenia)} klientów.")

    # 3. Dodawanie danych kontaktowych
    print("Rozpoczęcie aktualizacji danych kontaktowych...")
    try:
        imiona_men_df = pd.read_csv('../data/imiona_men.csv')
        nazwiska_men_df = pd.read_csv('../data/nazwiska_men.csv')
        imiona_woman_df = pd.read_csv('../data/imiona_woman.csv')
        nazwiska_woman_df = pd.read_csv('../data/nazwiska_woman.csv')

        # Zakładam, że pliki CSV mają kolumny 'imie' i 'nazwisko'
        men_names = imiona_men_df['imie'].tolist()
        men_surnames = nazwiska_men_df['nazwisko'].tolist()
        women_names = imiona_woman_df['imie'].tolist()
        women_surnames = nazwiska_woman_df['nazwisko'].tolist()

    except FileNotFoundError as e:
        print(f"Błąd: Nie znaleziono pliku {e.filename}")
        exit()
    except KeyError as e:
        print(f"Błąd: Brak wymaganej kolumny {e} w pliku CSV")
        exit()

    relations = ["mother", "father", "sister", "brother", "friend", "partner", "aunt", "uncle"]
    relation_weights = [20, 20, 10, 10, 25, 5, 5, 5]

    def losowy_numer():
        return ''.join(str(random.randint(0, 9)) for _ in range(9))

    def generuj_kontakt(relacja, nazwisko_klienta):
        if relacja in ["mother", "sister", "aunt"]:
            imie = random.choice(women_names)
            nazwisko = nazwisko_klienta if relacja in ["mother", "sister"] else random.choice(women_surnames)
        elif relacja in ["father", "brother", "uncle"]:
            imie = random.choice(men_names)
            nazwisko = nazwisko_klienta if relacja in ["father", "brother"] else random.choice(men_surnames)
        else:  # friend, partner
            imie = random.choice(women_names + men_names)
            nazwisko = random.choice(women_surnames + men_surnames)
        return f"{imie} {nazwisko}"

    cursor.execute("SELECT pesel, last_name FROM Customers")
    rows = cursor.fetchall()

    if not rows:
        print("Brak klientów w bazie danych. Najpierw dodaj klientów.")
        exit()

    dane_do_wstawienia = []
    for pesel, last_name in rows:
        phone = f"{losowy_numer()}"
        emergency_phone = f"+48{losowy_numer()}"
        relacja = random.choices(relations, weights=relation_weights, k=1)[0]
        kontakt = generuj_kontakt(relacja, last_name)
        dane_do_wstawienia.append((phone, kontakt, emergency_phone, relacja, pesel))

    sql = """
        UPDATE Customers
        SET phone_number = %s,
            emergency_contact_name = %s,
            emergency_contact_phone = %s,
            emergency_contact_relation = %s
        WHERE pesel = %s
    """
    cursor.executemany(sql, dane_do_wstawienia)
    con.commit()
    print(f"Zaktualizowano dane kontaktowe dla {len(dane_do_wstawienia)} klientów.")

except mysql.connector.Error as err:
    print(f"Błąd MySQL: {err}")
finally:
    if 'con' in locals() and con.is_connected():
        cursor.close()
        con.close()
        print("Połączenie z bazą danych zostało zamknięte.")

Rozpoczęcie aktualizacji emaili...
Zaktualizowano emaile dla 674 klientów.
Rozpoczęcie aktualizacji kolumny is_insured...
Zaktualizowano kolumnę is_insured dla 674 klientów.
Rozpoczęcie aktualizacji danych kontaktowych...
Zaktualizowano dane kontaktowe dla 674 klientów.
Połączenie z bazą danych zostało zamknięte.


## TripCrew

In [14]:
import random
import mysql.connector
from datetime import datetime

def assign_crew_to_trips():
    try:
        con = mysql.connector.connect(
            host="giniewicz.it",
            user="team07",
            password="te@mlot",
            database="team07",
            charset="utf8"
        )
        cursor = con.cursor(dictionary=True)

        print("Rozpoczęcie przypisywania załogi do wycieczek...")

        # Czyszczenie istniejących przypisań (opcjonalne)
        cursor.execute("SET FOREIGN_KEY_CHECKS = 0;")
        cursor.execute("DELETE FROM TripCrew")
        cursor.execute("ALTER TABLE TripCrew AUTO_INCREMENT = 1")
        cursor.execute("SET FOREIGN_KEY_CHECKS = 1;")
        con.commit()

        # Pobierz wszystkie wycieczki z datami
        cursor.execute("""
            SELECT trip_id, launch_date, return_date
            FROM Trips
            ORDER BY trip_id
        """)
        trips = cursor.fetchall()

        # Pobierz wszystkich pracowników
        cursor.execute("SELECT pesel FROM Employees")
        employees = [row['pesel'] for row in cursor.fetchall()]
        if len(employees) < 2 * len(trips):
            print(f"Za mało pracowników ({len(employees)}). Potrzeba co najmniej {2 * len(trips)}.")
            return

        # Inicjalizacja listy zajętych pilotów i ich dat powrotu
        busy_pilots = {}  

        for trip in trips:
            trip_id = trip['trip_id']
            departure_date = trip['launch_date']
            return_date = trip['return_date']

            # Wybierz pilota (bez konfliktów)
            available_pilots = [emp for emp in employees if emp not in busy_pilots or busy_pilots[emp] < departure_date]
            if not available_pilots:
                print(f"Brak dostępnych pilotów dla trip_id {trip_id}. Pomijam.")
                continue
            pilot = random.choice(available_pilots)
            busy_pilots[pilot] = return_date  # Zablokuj pilota do daty powrotu

            # Wybierz drugiego pracownika (np. co-pilot), unikając powtórzenia pilota
            available_crew = [emp for emp in employees if emp != pilot and (emp not in busy_pilots or busy_pilots[emp] < departure_date)]
            if not available_crew:
                print(f"Brak dostępnego co-pilota dla trip_id {trip_id}. Pomijam.")
                continue
            crew_member = random.choice(available_crew)

            # Wstaw do TripCrew
            cursor.execute("""
                INSERT INTO TripCrew (trip_id, employee_id, role)
                VALUES (%s, %s, %s)
            """, (trip_id, pilot, 'pilot'))
            cursor.execute("""
                INSERT INTO TripCrew (trip_id, employee_id, role)
                VALUES (%s, %s, %s)
            """, (trip_id, crew_member, 'co-pilot'))

            print(f"Przypisano pilota {pilot} i co-pilota {crew_member} do trip_id {trip_id}.")

        con.commit()
        print(f"Pomyślnie przypisano załogę do {len(trips)} wycieczek.")

    except mysql.connector.Error as err:
        print(f"Błąd podczas operacji na bazie danych: {err}")
    finally:
        if 'con' in locals() and con.is_connected():
            cursor.close()
            con.close()
            print("Połączenie z bazą danych zostało zamknięte.")

if __name__ == "__main__":
    assign_crew_to_trips()

Rozpoczęcie przypisywania załogi do wycieczek...
Przypisano pilota 24410396508 i co-pilota 27420183258 do trip_id 1.
Przypisano pilota 43492192729 i co-pilota 45440102761 do trip_id 2.
Przypisano pilota 33511408955 i co-pilota 19440360731 do trip_id 3.
Przypisano pilota 45441618030 i co-pilota 27420183258 do trip_id 4.
Przypisano pilota 42462891154 i co-pilota 28491502902 do trip_id 5.
Przypisano pilota 45412318141 i co-pilota 43510767939 do trip_id 6.
Przypisano pilota 27501408533 i co-pilota 39462867274 do trip_id 7.
Przypisano pilota 28502273050 i co-pilota 42482877002 do trip_id 8.
Przypisano pilota 37412241622 i co-pilota 42501574448 do trip_id 9.
Przypisano pilota 35411047621 i co-pilota 31521614542 do trip_id 10.
Przypisano pilota 23411578524 i co-pilota 34442084618 do trip_id 11.
Przypisano pilota 22470218488 i co-pilota 30522909899 do trip_id 12.
Przypisano pilota 28502273050 i co-pilota 19451206169 do trip_id 13.
Przypisano pilota 40471029368 i co-pilota 17511335244 do trip_i

## Naprawa rakiet

In [15]:
import random
import mysql.connector
from datetime import datetime, timedelta

def assign_recovery_tasks():
    try:
        con = mysql.connector.connect(
            host="giniewicz.it",
            user="team07",
            password="te@mlot",
            database="team07",
            charset="utf8"
        )
        cursor = con.cursor(dictionary=True)

        print("Rozpoczęcie przypisywania zadań odzysku...")

        # Czyszczenie istniejących zadań (opcjonalne)
        cursor.execute("SET FOREIGN_KEY_CHECKS = 0;")
        cursor.execute("DELETE FROM RecoveryTasks")
        cursor.execute("ALTER TABLE RecoveryTasks AUTO_INCREMENT = 1")
        cursor.execute("SET FOREIGN_KEY_CHECKS = 1;")
        con.commit()

        # Pobierz wszystkie wycieczki z trip_id, rocket_id (spaceship_id) i return_date
        cursor.execute("""
            SELECT trip_id, rocket_id, return_date
            FROM Trips
            ORDER BY trip_id
        """)
        trips = cursor.fetchall()

        # Pobierz pracowników ze specjalizacją 'rockets'
        cursor.execute("SELECT pesel FROM Employees WHERE specialization = 'rockets'")
        employees = [row['pesel'] for row in cursor.fetchall()]
        if not employees:
            print("Brak pracowników ze specjalizacją 'rockets'.")
            return

        # Inicjalizacja listy zajętych pracowników (bez blokady na powtórzenia)
        busy_employees = {}  # Śledzi tylko ostatnie end_date dla informacji

        for trip in trips:
            trip_id = trip['trip_id']
            rocket_id = trip['rocket_id']
            return_date = trip['return_date']
            start_date = return_date
            end_date = start_date + timedelta(days=random.randint(27, 30))

            # Wybierz dowolnego pracownika (powtarzalnego)
            if not employees:
                print(f"Brak pracowników dla trip_id {trip_id}. Pomijam.")
                continue
            employee = random.choice(employees)
            busy_employees[employee] = end_date  # Aktualizuj ostatnią datę zakończenia

            # Losowy status
            status = random.choice(['pending', 'in_progress'])

            # Wstaw zadanie
            cursor.execute("""
                INSERT INTO RecoveryTasks (pesel, rocket_id, start_date, end_date, status)
                VALUES (%s, %s, %s, %s, %s)
            """, (employee, rocket_id, start_date, end_date, status))

            print(f"Przypisano zadanie odzysku dla rocket_id {rocket_id} do pracownika {employee} (od {start_date} do {end_date}, status: {status}).")

        con.commit()
        print(f"Pomyślnie przypisano zadania odzysku do {len(trips)} wycieczek.")

    except mysql.connector.Error as err:
        print(f"Błąd podczas operacji na bazie danych: {err}")
    finally:
        if 'con' in locals() and con.is_connected():
            cursor.close()
            con.close()
            print("Połączenie z bazą danych zostało zamknięte.")

if __name__ == "__main__":
    assign_recovery_tasks()

Rozpoczęcie przypisywania zadań odzysku...
Przypisano zadanie odzysku dla rocket_id 12 do pracownika 17513025262 (od 2168-05-05 do 2168-06-03, status: in_progress).
Przypisano zadanie odzysku dla rocket_id 10 do pracownika 27421846363 (od 2168-02-04 do 2168-03-05, status: pending).
Przypisano zadanie odzysku dla rocket_id 1 do pracownika 17422887960 (od 2168-07-29 do 2168-08-28, status: pending).
Przypisano zadanie odzysku dla rocket_id 3 do pracownika 38452521901 (od 2168-06-04 do 2168-07-01, status: in_progress).
Przypisano zadanie odzysku dla rocket_id 4 do pracownika 38452521901 (od 2168-07-08 do 2168-08-04, status: pending).
Przypisano zadanie odzysku dla rocket_id 9 do pracownika 44442100050 (od 2168-06-04 do 2168-07-01, status: pending).
Przypisano zadanie odzysku dla rocket_id 8 do pracownika 20521122962 (od 2168-11-28 do 2168-12-28, status: pending).
Przypisano zadanie odzysku dla rocket_id 6 do pracownika 42462099699 (od 2170-01-05 do 2170-02-03, status: in_progress).
Przypis

In [16]:
import random
import mysql.connector
from datetime import datetime, timedelta

def assign_recovery_tasks():
    try:
        con = mysql.connector.connect(
            host="giniewicz.it",
            user="team07",
            password="te@mlot",
            database="team07",
            charset="utf8"
        )
        cursor = con.cursor(dictionary=True)

        print("Rozpoczęcie przypisywania zadań odzysku...")

        # Czyszczenie istniejących zadań (opcjonalne)
        cursor.execute("SET FOREIGN_KEY_CHECKS = 0;")
        cursor.execute("DELETE FROM RepairTasks")
        cursor.execute("ALTER TABLE RepairTasks AUTO_INCREMENT = 1")
        cursor.execute("SET FOREIGN_KEY_CHECKS = 1;")
        con.commit()

        # Pobierz wszystkie wycieczki z trip_id, spaceship_id) i return_date
        cursor.execute("""
            SELECT trip_id, spaceship_id, return_date
            FROM Trips
            ORDER BY trip_id
        """)
        trips = cursor.fetchall()

        # Pobierz pracowników ze specjalizacją 'rockets'
        cursor.execute("SELECT pesel FROM Employees WHERE specialization = 'spaceships'")
        employees = [row['pesel'] for row in cursor.fetchall()]
        if not employees:
            print("Brak pracowników ze specjalizacją 'spaceships'.")
            return

        # Inicjalizacja listy zajętych pracowników (bez blokady na powtórzenia)
        busy_employees = {}  # Śledzi tylko ostatnie end_date dla informacji

        for trip in trips:
            trip_id = trip['trip_id']
            spaceship_id = trip['spaceship_id']
            return_date = trip['return_date']
            start_date = return_date
            end_date = start_date + timedelta(days=random.randint(27, 30))

            # Wybierz dowolnego pracownika (powtarzalnego)
            if not employees:
                print(f"Brak pracowników dla trip_id {trip_id}. Pomijam.")
                continue
            employee = random.choice(employees)
            busy_employees[employee] = end_date  # Aktualizuj ostatnią datę zakończenia

            # Losowy status
            status = random.choice(['pending', 'in_progress'])

            # Wstaw zadanie
            cursor.execute("""
                INSERT INTO RepairTasks (employee_id, spaceship_id, start_date, end_date, status)
                VALUES (%s, %s, %s, %s, %s)
            """, (employee, spaceship_id, start_date, end_date, status))

            print(f"Przypisano zadanie odzysku dla spaceship_id {spaceship_id} do pracownika {employee} (od {start_date} do {end_date}, status: {status}).")

        con.commit()
        print(f"Pomyślnie przypisano zadania odzysku do {len(trips)} wycieczek.")

    except mysql.connector.Error as err:
        print(f"Błąd podczas operacji na bazie danych: {err}")
    finally:
        if 'con' in locals() and con.is_connected():
            cursor.close()
            con.close()
            print("Połączenie z bazą danych zostało zamknięte.")

if __name__ == "__main__":
    assign_recovery_tasks()

Rozpoczęcie przypisywania zadań odzysku...
Przypisano zadanie odzysku dla spaceship_id 9 do pracownika 26511469369 (od 2168-05-05 do 2168-06-02, status: in_progress).
Przypisano zadanie odzysku dla spaceship_id 2 do pracownika 45493048665 (od 2168-02-04 do 2168-03-05, status: in_progress).
Przypisano zadanie odzysku dla spaceship_id 11 do pracownika 31521614542 (od 2168-07-29 do 2168-08-26, status: pending).
Przypisano zadanie odzysku dla spaceship_id 10 do pracownika 25521581580 (od 2168-06-04 do 2168-07-02, status: in_progress).
Przypisano zadanie odzysku dla spaceship_id 12 do pracownika 19472227198 (od 2168-07-08 do 2168-08-06, status: pending).
Przypisano zadanie odzysku dla spaceship_id 1 do pracownika 36432378484 (od 2168-06-04 do 2168-07-03, status: pending).
Przypisano zadanie odzysku dla spaceship_id 9 do pracownika 38501866755 (od 2168-11-28 do 2168-12-27, status: pending).
Przypisano zadanie odzysku dla spaceship_id 7 do pracownika 24490912913 (od 2170-01-05 do 2170-02-04, 

# PSEUDOKOD SYMULACJI

1. Definiujemy nasz startowy budżet firmy 
2. Generujemy losową ścieżkę klientów w przedziale czasowym co pół roku.

while t < end_time:

if t == indeks[i] # kolejne indeksy, w których zostali wygenerowani klienci

1. Z Bazy Spaceships i Rockets wybiera odpowiednie statki i rakiety, o statusie active, albo patrzy na date naprawy w sewisie. tak by było dla wystarczającej liczby klientów. Dla tych rakiet losuje jedną z kompatybilnych wycieczek. Jeśli klientów jest za dużo obsługuje ile może.

2. Za pomocą naszej funkcji solar liczy dystans(do obliczenia kosztów) i czas (do daty powrotu) prędkość z statku.

3. Koszt organizacji: liczba uczestników/(liczba klientów w tym roku) * SUM(salaries) + koszt rakiety * 0.7 + koszt statku * 0.15 + koszt pobytu na miejscu base_cost

4. Dodanie zespołu pilotów(1 stacja kosmiczna 2 księżyc 3 międzyplanetarni) i obsługi (10% pojemności min 2)  z tablicy Employees według kategorii

5. Wygenerowanie klientów tak jak pracowników  i "wsadzenie ich do statku"
6. przekazanie wszystkich danych do trips
7. przekazanie rakiety do serwisu i przypisanie do niejj pracowników
Pętla przechodząca przez tabele serwisów statków i serwisów rakiet:
	zmienia stan na active jeśli termin został osiągnięty pobiera opłate za dopłatę do naprawy (serwis odzyskuje 70% musimy pokryć 30% initial cost)
if t == 365
	Wypłata pensji




## WYGENEROWANIE Klientów

1. Nowych klientów tworzymy tak samo jak pracowników wyżej.
2. Starzy klienci: Najpierw sprawdza czy już byli na tej wycieczce. I zależnie jak im się podobało to mogą wrócić
   Starzy klienci, którzy nie byli: też mogą spróbować, ale z mniejszą szansą
3. Proporcje im droższa wycieczka tym większa szansa na starych klientów względem nowych.
3. Koszt biletu: Dzieli koszta organizacji / liczbe uczestników * 1.2 + SUM(salaries) * liczba uczestników/(liczba klientów w tym roku) / liczbe uczestników Koszta organizacji

## Symulacja mechaników